In [1]:
from pathlib import Path
import json
import pandas as pd
import duckdb

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORT_DIR = PROJECT_ROOT / "reports"

con = duckdb.connect()

historical_labels = con.read_parquet(
    str(PROCESSED_DIR / "historical_labels_dec2016.parquet")
).df()

temporal_validation_data = con.read_parquet(
    str(
        PROCESSED_DIR
        / "candidate_transaction_features_v2_cutoff_20170228.parquet"
    )
).df()

label_config = json.loads(
    (REPORT_DIR / "historical_label_config.json").read_text(
        encoding="utf-8"
    )
)

validation_config = json.loads(
    (REPORT_DIR / "transaction_feature_config.json").read_text(
        encoding="utf-8"
    )
)

TRAIN_CUTOFF = pd.Timestamp(label_config["history_cutoff"])
VALIDATION_CUTOFF = pd.Timestamp(validation_config["cutoff"])

assert pd.Timestamp(
    label_config["outcome_observation_end"]
) < VALIDATION_CUTOFF

print("Training feature cutoff:", TRAIN_CUTOFF.date())
print("Validation feature cutoff:", VALIDATION_CUTOFF.date())

Training feature cutoff: 2016-11-30
Validation feature cutoff: 2017-02-28


In [2]:
feature_sql = (
    PROJECT_ROOT / "sql" / "transaction_features_90d.sql"
).read_text(encoding="utf-8")

con.register("cohort_users", historical_labels[["msno"]])

train_window_start = TRAIN_CUTOFF - pd.Timedelta(days=89)

historical_features = con.execute(
    feature_sql,
    [
        str(RAW_DIR / "transactions.csv"),
        train_window_start.date(),
        TRAIN_CUTOFF.date(),
        TRAIN_CUTOFF.date(),
    ],
).df()

historical_training_data = historical_labels.merge(
    historical_features,
    on="msno",
    how="left",
    validate="one_to_one",
    indicator="transaction_join",
)

historical_training_data["has_transaction_90d"] = (
    historical_training_data["transaction_join"] == "both"
).astype("int8")

historical_training_data = historical_training_data.drop(
    columns="transaction_join"
)

count_columns = [
    "transaction_rows_90d",
    "transaction_days_90d",
    "cancel_rows_90d",
]

historical_training_data[count_columns] = (
    historical_training_data[count_columns]
    .fillna(0)
    .astype("int64")
)

print("Historical feature window:", train_window_start.date(), TRAIN_CUTOFF.date())

Historical feature window: 2016-09-02 2016-11-30


In [3]:
assert len(historical_training_data) == len(historical_labels)
assert historical_training_data["msno"].is_unique
assert temporal_validation_data["msno"].is_unique

feature_columns = [
    "transaction_rows_90d",
    "transaction_days_90d",
    "cancel_rows_90d",
    "auto_renew_share_90d",
    "avg_plan_days_90d",
    "days_since_last_transaction_90d",
    "has_transaction_90d",
]

for dataset in [historical_training_data, temporal_validation_data]:
    assert dataset["is_churn"].isin([0, 1]).all()
    assert set(feature_columns).issubset(dataset.columns)

cohort_summary = pd.DataFrame([
    {
        "cohort": "Historical training",
        "users": len(historical_training_data),
        "churn_rate": historical_training_data["is_churn"].mean(),
        "transaction_coverage": (
            historical_training_data["has_transaction_90d"].mean()
        ),
    },
    {
        "cohort": "Later validation",
        "users": len(temporal_validation_data),
        "churn_rate": temporal_validation_data["is_churn"].mean(),
        "transaction_coverage": (
            temporal_validation_data["has_transaction_90d"].mean()
        ),
    },
])

shared_user_count = historical_training_data["msno"].isin(
    temporal_validation_data["msno"]
).sum()

print("Users appearing in both cohorts:", shared_user_count)
cohort_summary

Users appearing in both cohorts: 736338


,cohort,users,churn_rate,transaction_coverage
0,Historical training,882127,0.096627,0.979928
1,Later validation,970960,0.089942,0.971388


## Train on Historical Data and Evaluate on the Later Cohort

In [4]:
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    log_loss,
    roc_auc_score,
    average_precision_score,
)

X_history = historical_training_data[feature_columns]
y_history = historical_training_data["is_churn"].astype("int8")

X_later = temporal_validation_data[feature_columns]
y_later = temporal_validation_data["is_churn"].astype("int8")

temporal_model = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median", add_indicator=True),
    ),
    ("scaler", StandardScaler()),
    (
        "classifier",
        LogisticRegression(max_iter=1000, random_state=42),
    ),
])

temporal_model.fit(X_history, y_history)

later_probabilities = temporal_model.predict_proba(X_later)[:, 1]

assert np.isfinite(later_probabilities).all()
assert (
    (later_probabilities >= 0)
    & (later_probabilities <= 1)
).all()

print("Historical model fitted and later cohort scored.")

Historical model fitted and later cohort scored.


In [5]:
def evaluate_probabilities(name, y_true, probabilities):
    return {
        "model": name,
        "log_loss": log_loss(y_true, probabilities),
        "roc_auc": roc_auc_score(y_true, probabilities),
        "average_precision": average_precision_score(
            y_true, probabilities
        ),
    }

historical_constant_probabilities = np.full(
    len(y_later),
    y_history.mean(),
)

historical_availability_model = LogisticRegression(
    max_iter=1000,
    random_state=42,
)

historical_availability_model.fit(
    X_history[["has_transaction_90d"]],
    y_history,
)

availability_probabilities = (
    historical_availability_model.predict_proba(
        X_later[["has_transaction_90d"]]
    )[:, 1]
)

temporal_results = pd.DataFrame([
    evaluate_probabilities(
        "Historical constant probability",
        y_later,
        historical_constant_probabilities,
    ),
    evaluate_probabilities(
        "Historical availability-only model",
        y_later,
        availability_probabilities,
    ),
    evaluate_probabilities(
        "Historical logistic regression",
        y_later,
        later_probabilities,
    ),
]).sort_values("log_loss").reset_index(drop=True)

temporal_results

,model,log_loss,roc_auc,average_precision
0,Historical logistic regression,0.212926,0.82958,0.536523
1,Historical availability-only model,0.255099,0.63069,0.289206
2,Historical constant probability,0.302665,0.50000,0.089942


In [6]:
seen_in_training = temporal_validation_data["msno"].isin(
    historical_training_data["msno"]
).to_numpy()

segment_results = []

for segment_name, mask in [
    ("Seen in historical training", seen_in_training),
    ("Not seen in historical training", ~seen_in_training),
]:
    segment_labels = y_later.loc[mask]
    segment_probabilities = later_probabilities[mask]

    result = {
        "segment": segment_name,
        "users": int(mask.sum()),
        "churn_rate": float(segment_labels.mean()),
        "log_loss": log_loss(
            segment_labels,
            segment_probabilities,
            labels=[0, 1],
        ),
    }

    if segment_labels.nunique() == 2:
        result["roc_auc"] = roc_auc_score(
            segment_labels, segment_probabilities
        )
        result["average_precision"] = average_precision_score(
            segment_labels, segment_probabilities
        )
    else:
        result["roc_auc"] = np.nan
        result["average_precision"] = np.nan

    segment_results.append(result)

temporal_segment_results = pd.DataFrame(segment_results)
temporal_segment_results

,segment,users,churn_rate,log_loss,roc_auc,average_precision
0,Seen in historical training,736338,0.046605,0.168146,0.718504,0.182890
1,Not seen in historical training,234622,0.225951,0.353464,0.872967,0.713922


In [7]:
temporal_results.to_csv(
    REPORT_DIR / "temporal_model_comparison.csv",
    index=False,
)

temporal_segment_results.to_csv(
    REPORT_DIR / "temporal_segment_results.csv",
    index=False,
)

cohort_summary.to_csv(
    REPORT_DIR / "temporal_cohort_summary.csv",
    index=False,
)

temporal_config = {
    "evaluation_method": "historical cohort evaluation",
    "training_label_source": "reconstructed December 2016 labels",
    "validation_label_source": "train_v2.csv",
    "transaction_source": "transactions.csv",
    "training_feature_cutoff": str(TRAIN_CUTOFF.date()),
    "training_outcome_end": label_config["outcome_observation_end"],
    "validation_feature_cutoff": str(VALIDATION_CUTOFF.date()),
    "window_days": 90,
    "training_users": len(y_history),
    "validation_users": len(y_later),
    "shared_users": int(shared_user_count),
    "features": feature_columns,
    "preprocessing_fit_on": "historical training data only",
    "training_outcomes_end_before_validation_cutoff": True,
    "official_label_parity_verified": False,
    "validation_label_period_independently_verified": False,
    "point_in_time_source_snapshots_verified": False,
    "independent_test_evaluation_completed": False,
}

(REPORT_DIR / "temporal_evaluation_config.json").write_text(
    json.dumps(temporal_config, indent=2) + "\n",
    encoding="utf-8",
)

print("Historical cohort evaluation reports saved.")

Historical cohort evaluation reports saved.
